In [17]:
## IMPORTING LIBRARIES REQUIRED FOR PROJECT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [18]:
##UPLOAD THE DATASET 
df=pd.read_csv('../Dataset/cardekho_dataset.csv')
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [19]:
##LOOKING FOR NULL VALUES 
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [20]:
df['seller_type'].unique()

array(['Individual', 'Dealer', 'Trustmark Dealer'], dtype=object)

In [21]:
##DROP EXTRA COLUMNS SO THAT DATASET BECOME PERFECT FOR TRAINING
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)

In [22]:
##FINDING UNIQUE VALUE FROM THE COLUMN MODEL
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [23]:
##GETTING ALL DIFFERENT TYPE OF FEATURE FOR BETTER UNDERSTANDING OF DATASET
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Number of numerical variables: ', len(num_features))

Number of numerical variables:  8


In [24]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Number of categorical variables: ', len(cat_features))

Number of categorical variables:  4


In [25]:
discrete_features=[feature for feature in num_features if len(df[feature].unique())<25]
print("Number of discrete variables :",len(discrete_features))

Number of discrete variables : 2


In [26]:
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print("Number of continuous variables :",len(continuous_features))

Number of continuous variables : 6


In [27]:
##IMPORT TRAIN TEST AND SELECTING X AND Y VALUES
from sklearn.model_selection import train_test_split
X=df.drop('selling_price',axis=1)
y=df['selling_price']

In [28]:
##FINDING CATEGORIES IN SELLER , FUEL AND TRANSMISSION COLUMN
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [29]:
##IMPORTING COLUMN TRANSFORMER
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


categorical_ordinal = ['model'] 
categorical_onehot = ['seller_type', 'fuel_type', 'transmission_type']
numerical_features = X.select_dtypes(exclude='object').columns

##CREATING TRANSFORMER

ordinal_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
onehot_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')
numeric_transformer = StandardScaler()

## Preprocessor (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', ordinal_transformer, categorical_ordinal),
        ('onehot', onehot_transformer, categorical_onehot),
        ('num', numeric_transformer, numerical_features)
    ],
    remainder='passthrough'
)

In [30]:
##SPLITING THE DATASET
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape

((12328, 11), (3083, 11))

In [40]:
##MPORTING DIFFERENT ML MODEL SO THAT WE CAN FIND BETTER ONE TO PREDICT 
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [32]:
## CREATING A FUNCTION FOR EVALUTION OF MODELS
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2=r2_score(true,predicted)
    return mae,mse,rmse,r2

In [33]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [34]:
##TRAINIG AND TESTING DIFFERENT MODEL TO FIND BEST ONE
models ={
    'LinearRegression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'KNeighborsRegressor':KNeighborsRegressor(),
    'DecisionTreeRegressor':DecisionTreeRegressor(),
    'RandomForestRegressor':RandomForestRegressor()
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train_encoded,y_train)


    y_train_pred=model.predict(X_train_encoded)
    y_test_pred=model.predict(X_test_encoded)

    model_train_mae,model_train_mse,model_train_rmse,model_train_r2=evaluate_model(y_train,y_train_pred)

    model_test_mae,model_test_mse,model_test_rmse,model_test_r2=evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])
## PRINTING EVALUTION METRICES FOR EVERY MODEL 
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))
    print('- MSE {:.4f}'.format(model_train_mse))

    print('----------------------------------')

    print('Model performance for Testing set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    print('-MSE {:.4f}'.format(model_test_mse))

    print('----------------------------------')

LinearRegression
Model performance for Training set
- Root Mean Squared Error: 553863.7987
- Mean Absolute Error: 268081.7682
- R2 Score: 0.6218
- MSE 306765107557.3501
----------------------------------
Model performance for Testing set
- Root Mean Squared Error: 502449.7465
- Mean Absolute Error: 279638.4935
- R2 Score: 0.6646
-MSE 252455747747.6377
----------------------------------
Lasso
Model performance for Training set
- Root Mean Squared Error: 553863.8032
- Mean Absolute Error: 268079.5149
- R2 Score: 0.6218
- MSE 306765112444.0403
----------------------------------
Model performance for Testing set
- Root Mean Squared Error: 502448.9219
- Mean Absolute Error: 279634.7455
- R2 Score: 0.6646
-MSE 252454919137.0536
----------------------------------
Ridge
Model performance for Training set
- Root Mean Squared Error: 553864.4428
- Mean Absolute Error: 268039.6417
- R2 Score: 0.6218
- MSE 306765821049.6565
----------------------------------
Model performance for Testing set
- Root

In [35]:
''' WE HAVE FIND THE BEST  AMONG OTHER SO THAT NOW WE TUNE
 ITS HYPERPARAMETER TO GET MORE PERFECT OUTPUT SO WE WILL TRY 
 TO FIND PERFECT VALUE FOR PARAMETER SO WE WILL GET MORE ACCURATE'''

rf_params= {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 8, 10,15],
    'min_samples_split': [2, 8, 15, 20],
    "max_features":['auto',5,7,8]
}

In [36]:
##PASSING THE PARAMETER VALUE OF THE MODEL
randomcv_models = [
                   ('RandomForestRegressor',RandomForestRegressor(),rf_params)]

In [38]:
## IMPORTING RANDOMIZED SEARCH CV TO FIND PERFECT VALUE FOR THE HYPER PARAMETER
from sklearn.model_selection import RandomizedSearchCV
model_params = {} 

random=RandomizedSearchCV(estimator=model,param_distributions=rf_params,n_iter=100,cv=5,verbose=2,n_jobs=-1)
random.fit(X_train_encoded,y_train)
random.best_params_



Fitting 5 folds for each of 100 candidates, totalling 500 fits


{'n_estimators': 300,
 'min_samples_split': 2,
 'max_features': 8,
 'max_depth': None}

In [41]:
## AS WE GET BEST PARAMETER VALUE SO WE WILL PUT AND CHECK THE RESULTS

model=RandomForestRegressor(n_estimators= 100, min_samples_split= 2, max_features= 7, max_depth= None)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model)
])

pipeline.fit(X_train,y_train)
y_train2=pipeline.predict(X_train)
print(y_train2)


# Test it quickly
score = pipeline.score(X_test, y_test)
print(f"Final Test Score (R2): {score:.4f}")


[1985850.          528666.66666667 7006000.         ...  280150.
  637030.          948460.        ]
Final Test Score (R2): 0.9397


In [44]:
import joblib

joblib.dump(pipeline, '../Models/car_prediction_model.pkl', compress=3)


['../Models/car_prediction_model.pkl']